# 03 — Data Cleaning & Quality Audit

**Project:** Predicting Corporate GHG Intensity  
**Purpose:** Verify dataset integrity and inspect the missingness patterns of variables.

---


In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

### 1. Load Linked Panel

In [2]:
df = pd.read_csv(INTERIM_DIR / "linked_panel.csv")
print("Linked Panel Shape:", df.shape)
df.head()

Linked Panel Shape: (2791, 23)


,cik,ticker,company_name,sector,sic_code,year,total_assets,revenue,net_income,operating_income,...,stockholders_equity,scope1_emissions,co2_non_biogenic,n_facilities,primary_naics,high_emission_naics,selected,us_gdp_growth,us_co2_per_capita,us_energy_use_per_capita
0,19871,CVR,Chicago Rivet & Machine Co.,Manufacturing,3540,2018,33246625.0,37174249.0,2001185.0,2402648.0,...,29759749.0,55180.90,168.4,2.0,562212.0,0,1,2.966505,15.2,6738.272724
1,19871,CVR,Chicago Rivet & Machine Co.,Manufacturing,3540,2019,31723376.0,32873002.0,538314.0,491584.0,...,29158027.0,30150.65,246.9,2.0,562212.0,0,1,2.583825,14.8,6700.930340
2,19871,CVR,Chicago Rivet & Machine Co.,Manufacturing,3540,2020,31238071.0,27590653.0,50450.0,-83014.0,...,28706089.0,NaN,NaN,NaN,NaN,0,0,-2.163029,13.0,6141.531258
3,19871,CVR,Chicago Rivet & Machine Co.,Manufacturing,3540,2021,31766258.0,33974558.0,1113472.0,1358915.0,...,28969365.0,NaN,NaN,NaN,NaN,0,0,6.055053,13.9,6445.834092
4,19871,CVR,Chicago Rivet & Machine Co.,Manufacturing,3540,2022,33626127.0,33646033.0,2867629.0,3561196.0,...,30986798.0,NaN,NaN,NaN,NaN,0,0,2.512375,13.6,6511.688414


### 2. Audit Missing Values & Logical Consistency
We inspect the count of missing values and check for invalid entries (e.g. assets <= 0).

In [3]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nFirms with non-positive assets:", (df["total_assets"] <= 0).sum())
print("Firms with non-positive revenue:", (df["revenue"] <= 0).sum())

Missing values per column:
cik                            0
ticker                         1
company_name                   0
sector                         0
sic_code                       0
year                           0
total_assets                   0
revenue                        0
net_income                   218
operating_income             278
capex                        827
rd_expense                     0
total_debt                  1133
stockholders_equity          204
scope1_emissions            1007
co2_non_biogenic            1007
n_facilities                1007
primary_naics               1007
high_emission_naics            0
selected                       0
us_gdp_growth                  0
us_co2_per_capita              0
us_energy_use_per_capita       0
dtype: int64

Firms with non-positive assets: 1
Firms with non-positive revenue: 58


### 3. Check Selection Bias Patterns
We audit reporting status across sectors.

In [4]:
report_rate = df.groupby("sector")["selected"].mean() * 100
print("EPA Reporting Rate (%) by Sector:")
print(report_rate.round(2))

EPA Reporting Rate (%) by Sector:
sector
Agriculture       56.00
Construction      40.62
Financials        51.08
Manufacturing     67.10
Mining            66.97
Retail            39.36
Services          42.74
Transportation    75.44
Utilities         95.62
Wholesale         64.20
Name: selected, dtype: float64


### Discussion & Next Steps
The data audit confirms that emissions data (`scope1_emissions`) is missing for most non-selected companies. This non-random missingness pattern confirms the presence of selection bias, validating the necessity of using the Heckman Selection correction. Next, we explore the data visually.